# LeetCode 563 — Binary Tree Tilt bằng Python

**Link đề:** https://leetcode.com/problems/binary-tree-tilt/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Binary Tree, DP on Trees

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findTilt(self, root: Optional[TreeNode]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **độ nghiêng (tilt)** của một node: `|tổng cây con trái − tổng cây con phải|`.
2. Ôn `TreeNode`, dựng cây từ list level order, đệ quy tính **tổng cây con**.
3. Thấy nút thắt của baseline (tính lại tổng nhiều lần, `O(n²)` với cây lệch) và rút ra mấu chốt: **duyệt hậu thứ tự (post-order) một lần**, hàm trả về tổng, tiện tay cộng độ nghiêng.
4. Viết thêm bản lặp bằng stack, trace call stack, chứng minh đúng, test đối chiếu.

## 1. Đề bài

Cho `root` của cây nhị phân, trả về **tổng độ nghiêng của mọi node**.

**Độ nghiêng** của một node = giá trị tuyệt đối của hiệu giữa **tổng mọi giá trị trong cây con trái** và **tổng mọi giá trị trong cây con phải**. Không có con trái thì tổng bên trái là 0 (tương tự bên phải).

```text
Ví dụ 2:        4
              /   \
             2     9
            / \     \
           3   5     7
```

| Node | Tổng trái | Tổng phải | Tilt |
|---:|---:|---:|---:|
| 3, 5, 7 (lá) | 0 | 0 | 0 |
| 2 | 3 | 5 | 2 |
| 9 | 0 | 7 | 7 |
| 4 | 3+5+2 = 10 | 9+7 = 16 | 6 |

Tổng: `0 + 0 + 0 + 2 + 7 + 6 = 15`.

| `root` | Output |
|---|---:|
| `[1,2,3]` | 1 |
| `[4,2,9,3,5,null,7]` | 15 |
| `[21,7,14,1,1,2,2,3,3]` | 9 |

**Constraints:** số node trong `[0, 10^4]`; `-1000 <= Node.val <= 1000`.

Chú ý: "tổng cây con trái" là tổng **mọi** node bên trái (cả cháu, chắt), **không** chỉ giá trị con trái trực tiếp. Cây có thể rỗng → 0.

## 2. Nền tảng: `TreeNode`, dựng cây, `abs`

Ta tự định nghĩa `TreeNode` và hàm `build_tree` dựng cây từ list level order (`None` = vị trí trống), dùng queue `deque` (`append` thêm cuối, `popleft` lấy đầu).

In [1]:
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left: "Optional[TreeNode]" = None, right: "Optional[TreeNode]" = None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def draw(node: Optional[TreeNode], depth: int = 0) -> None:
    # In cây xoay 90 độ: nhánh phải ở trên, nhánh trái ở dưới
    if node is None:
        return
    draw(node.right, depth + 1)
    print("    " * depth + str(node.val))
    draw(node.left, depth + 1)

null = None
ex2 = build_tree([4, 2, 9, 3, 5, null, 7])
draw(ex2)

        7
    9
4
        5
    2
        3


`abs(x)` trả về giá trị tuyệt đối: `abs(-6) = 6`, `abs(6) = 6`. Độ nghiêng không phụ thuộc bên nào lớn hơn.

In [2]:
print(abs(10 - 16), abs(16 - 10), abs(0 - 0))

6 6 0


### 2.1. Đệ quy tính tổng cây con

`subtree_sum(node)`:
- Base case: `None` → 0.
- Còn lại: `node.val + subtree_sum(left) + subtree_sum(right)`.

In [3]:
def subtree_sum(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return node.val + subtree_sum(node.left) + subtree_sum(node.right)

print("tổng cả cây:", subtree_sum(ex2))            # 4+2+9+3+5+7 = 30
print("tổng cây con trái của 4:", subtree_sum(ex2.left))   # 10
print("tổng cây con phải của 4:", subtree_sum(ex2.right))  # 16

tổng cả cây: 30
tổng cây con trái của 4: 10
tổng cây con phải của 4: 16


## 3. Cách 1 — Baseline: tại mỗi node gọi `subtree_sum` hai lần

Dịch thẳng định nghĩa:
1. Duyệt mọi node (đệ quy).
2. Tilt của node = `abs(subtree_sum(left) - subtree_sum(right))`.
3. Cộng dồn.

In [4]:
def find_tilt_naive(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    own = abs(subtree_sum(root.left) - subtree_sum(root.right))
    return own + find_tilt_naive(root.left) + find_tilt_naive(root.right)

print(find_tilt_naive(build_tree([1, 2, 3])))                    # 1
print(find_tilt_naive(ex2))                                      # 15
print(find_tilt_naive(build_tree([21, 7, 14, 1, 1, 2, 2, 3, 3])))  # 9

1
15
9


### Nút thắt

Mỗi node bị cộng vào tổng **một lần cho mỗi tổ tiên** của nó. Với cây lệch (như linked list) `n` node, tổng công việc là `1 + 2 + ... + n = O(n²)`. Đếm số lần gọi để thấy:

In [5]:
calls = 0
def subtree_sum_counted(node):
    global calls
    calls += 1
    if node is None:
        return 0
    return node.val + subtree_sum_counted(node.left) + subtree_sum_counted(node.right)

def find_tilt_naive_counted(root):
    if root is None:
        return 0
    own = abs(subtree_sum_counted(root.left) - subtree_sum_counted(root.right))
    return own + find_tilt_naive_counted(root.left) + find_tilt_naive_counted(root.right)

def build_chain(n: int) -> Optional[TreeNode]:
    root = None
    for v in range(n, 0, -1):
        root = TreeNode(v, root, None)     # lệch trái
    return root

for n in [10, 100, 400]:
    calls = 0
    find_tilt_naive_counted(build_chain(n))
    print(f"chain n={n:>3}: số lần gọi subtree_sum = {calls}")

chain n= 10: số lần gọi subtree_sum = 110
chain n=100: số lần gọi subtree_sum = 10100
chain n=400: số lần gọi subtree_sum = 160400


`global calls` cho phép hàm **gán** biến `calls` ở cấp module. Số lần gọi tăng theo bình phương.

## 4. Insight — Một lần post-order: "trả về tổng, cộng dồn tilt"

Để tính tổng cây con của `node`, ta **đã cần** tổng cây con trái `L` và phải `R`. Mà tilt của `node` cũng chỉ cần `L` và `R`!

⇒ Một hàm `dfs(node)`:
- **Trả về** tổng cây con gốc `node` = `node.val + L + R` (cho cha dùng).
- **Tiện tay** cộng `abs(L - R)` vào biến ngoài `total`.

Con được xử lý trước cha → **post-order**. Mỗi node thăm đúng một lần → `O(n)`.

Gợi ý của LeetCode: *"What about postorder traversal, using values of left and right childs?"* — đúng ý này.

### 4.1. `nonlocal` — sửa biến của hàm bao ngoài

Trong hàm lồng, muốn **gán** biến của hàm ngoài phải khai báo `nonlocal`, nếu không Python coi đó là biến cục bộ mới → `UnboundLocalError`.

In [6]:
def demo():
    total = 0
    def add(x):
        nonlocal total
        total += x
    for x in [2, 7, 6]:
        add(x)
    return total

print(demo())   # 15

15


## 5. Cách 2 — DFS post-order một lần (lời giải chính)

```python
def dfs(node):
    if node is None:
        return 0                  # cây rỗng: tổng 0
    L = dfs(node.left)            # tổng cây con trái
    R = dfs(node.right)           # tổng cây con phải
    total += abs(L - R)           # tilt của node
    return node.val + L + R       # tổng cây con gốc node
```

In [7]:
def find_tilt_dfs(root: Optional[TreeNode]) -> int:
    total = 0

    def dfs(node: Optional[TreeNode]) -> int:
        nonlocal total
        if node is None:
            return 0
        left_sum = dfs(node.left)
        right_sum = dfs(node.right)
        total += abs(left_sum - right_sum)
        return node.val + left_sum + right_sum

    dfs(root)
    return total

print(find_tilt_dfs(build_tree([1, 2, 3])))
print(find_tilt_dfs(ex2))
print(find_tilt_dfs(build_tree([21, 7, 14, 1, 1, 2, 2, 3, 3])))
print(find_tilt_dfs(None))

1
15
9
0


## 6. Trace call stack

Cây ví dụ 2. Thứ tự hoàn thành (post-order): 3, 5, 2, 7, 9, 4.

| Node xong | `L` | `R` | tilt `\|L−R\|` | total sau | trả về (tổng cây con) |
|---:|---:|---:|---:|---:|---:|
| 3 | 0 | 0 | 0 | 0 | 3 |
| 5 | 0 | 0 | 0 | 0 | 5 |
| 2 | 3 | 5 | 2 | 2 | 10 |
| 7 | 0 | 0 | 0 | 2 | 7 |
| 9 | 0 | 7 | 7 | 9 | 16 |
| 4 | 10 | 16 | 6 | **15** | 30 |

In [8]:
def trace_tilt(root: Optional[TreeNode]) -> int:
    total = 0

    def dfs(node, level):
        nonlocal total
        if node is None:
            return 0
        pad = "  " * level
        print(f"{pad}vào {node.val}")
        L = dfs(node.left, level + 1)
        R = dfs(node.right, level + 1)
        tilt = abs(L - R)
        total += tilt
        print(f"{pad}xong {node.val}: L={L}, R={R}, tilt={tilt}, total={total}, trả {node.val + L + R}")
        return node.val + L + R

    dfs(root, 0)
    print("Tổng tilt =", total)
    return total

trace_tilt(ex2)

vào 4
  vào 2
    vào 3
    xong 3: L=0, R=0, tilt=0, total=0, trả 3
    vào 5
    xong 5: L=0, R=0, tilt=0, total=0, trả 5
  xong 2: L=3, R=5, tilt=2, total=2, trả 10
  vào 9
    vào 7
    xong 7: L=0, R=0, tilt=0, total=2, trả 7
  xong 9: L=0, R=7, tilt=7, total=9, trả 16
xong 4: L=10, R=16, tilt=6, total=15, trả 30
Tổng tilt = 15


15

In [9]:
# Ví dụ 3 có cả cháu: [21,7,14,1,1,2,2,3,3]
ex3 = build_tree([21, 7, 14, 1, 1, 2, 2, 3, 3])
draw(ex3)
trace_tilt(ex3)

        2
    14
        2
21
        1
    7
            3
        1
            3
vào 21
  vào 7
    vào 1
      vào 3
      xong 3: L=0, R=0, tilt=0, total=0, trả 3
      vào 3
      xong 3: L=0, R=0, tilt=0, total=0, trả 3
    xong 1: L=3, R=3, tilt=0, total=0, trả 7
    vào 1
    xong 1: L=0, R=0, tilt=0, total=0, trả 1
  xong 7: L=7, R=1, tilt=6, total=6, trả 15
  vào 14
    vào 2
    xong 2: L=0, R=0, tilt=0, total=6, trả 2
    vào 2
    xong 2: L=0, R=0, tilt=0, total=6, trả 2
  xong 14: L=2, R=2, tilt=0, total=6, trả 18
xong 21: L=15, R=18, tilt=3, total=9, trả 54
Tổng tilt = 9


9

Ở ví dụ 3: node `1` (con trái của 7) có hai con `3, 3` → tilt 0; node 7: `L = 1+3+3 = 7`, `R = 1` → tilt 6; node 14: `|2 − 2| = 0`; root 21: `L = 7+7+1 = 15`, `R = 14+2+2 = 18` → tilt 3. Tổng `6 + 3 = 9` ✔.

## 7. Code nộp LeetCode — DFS đệ quy

In [10]:
class Solution:
    def findTilt(self, root: Optional[TreeNode]) -> int:
        total = 0

        def dfs(node: Optional[TreeNode]) -> int:
            nonlocal total
            if node is None:
                return 0
            left_sum = dfs(node.left)
            right_sum = dfs(node.right)
            total += abs(left_sum - right_sum)
            return node.val + left_sum + right_sum

        dfs(root)
        return total

sol = Solution()
print(sol.findTilt(build_tree([1, 2, 3])), sol.findTilt(ex2), sol.findTilt(ex3))

1 15 9


### Biến thể không dùng `nonlocal`: trả về cặp

Hàm đệ quy có thể trả **tuple** `(tổng cây con, tổng tilt trong cây con)` — không cần biến ngoài, dễ test từng phần.

In [11]:
def find_tilt_pair(root: Optional[TreeNode]) -> int:
    def solve(node):
        if node is None:
            return 0, 0                         # (tổng, tilt)
        s_left, t_left = solve(node.left)
        s_right, t_right = solve(node.right)
        s = node.val + s_left + s_right
        t = t_left + t_right + abs(s_left - s_right)
        return s, t
    return solve(root)[1]

print(find_tilt_pair(ex2), find_tilt_pair(ex3))

15 9


## 8. Cách 3 — Post-order lặp bằng stack

Cây sâu tới `10^4` có thể vượt giới hạn đệ quy mặc định (~1000) của Python. Bản lặp dùng stack `(node, visited)`:
- Gặp lần đầu: đẩy lại `(node, True)` rồi đẩy hai con → hai con xong trước.
- Gặp lần hai: đọc tổng hai con từ dict `sums`, cộng tilt, lưu tổng của node.

In [12]:
def find_tilt_iterative(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    total = 0
    sums = {None: 0}
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if visited:
            L, R = sums[node.left], sums[node.right]
            total += abs(L - R)
            sums[node] = node.val + L + R
        else:
            stack.append((node, True))
            if node.right:
                stack.append((node.right, False))
            if node.left:
                stack.append((node.left, False))
    return total

print(find_tilt_iterative(ex2), find_tilt_iterative(ex3))
chain = build_chain(10000)
print("cây lệch 10000 node:", find_tilt_iterative(chain))

15 9
cây lệch 10000 node: 333333330000


In [13]:
import sys
print("Giới hạn đệ quy:", sys.getrecursionlimit())
try:
    find_tilt_dfs(chain)
except RecursionError:
    print("Bản đệ quy gặp RecursionError với cây lệch 10000 node -> dùng bản lặp hoặc sys.setrecursionlimit")

Giới hạn đệ quy: 1000
Bản đệ quy gặp RecursionError với cây lệch 10000 node -> dùng bản lặp hoặc sys.setrecursionlimit


Kiểm tra kết quả cây lệch bằng tay: chain `1 → 2 → ... → n` (mỗi node chỉ có con trái). Tilt của node `k` = tổng `k+1 + ... + n`. Tổng tilt = `Σ_{k=1}^{n} Σ_{j=k+1}^{n} j = Σ_j j·(j−1)`.

In [14]:
n = 10000
formula = sum(j * (j - 1) for j in range(1, n + 1))
print("công thức:", formula, "| bản lặp:", find_tilt_iterative(chain), "| khớp:", formula == find_tilt_iterative(chain))

công thức: 333333330000 | bản lặp: 333333330000 | khớp: True


### Code nộp LeetCode — bản lặp

In [15]:
class Solution:
    def findTilt(self, root: Optional[TreeNode]) -> int:
        total = 0
        sums = {None: 0}
        stack = [(root, False)] if root else []
        while stack:
            node, visited = stack.pop()
            if visited:
                L, R = sums[node.left], sums[node.right]
                total += abs(L - R)
                sums[node] = node.val + L + R
            else:
                stack.append((node, True))
                if node.right:
                    stack.append((node.right, False))
                if node.left:
                    stack.append((node.left, False))
        return total

print(Solution().findTilt(ex2))

15


## 9. Vì sao Cách 2 đúng?

**Invariant:** khi `dfs(node)` trả về,
1. giá trị trả về = tổng mọi giá trị trong cây con gốc `node`;
2. `total` đã cộng thêm **đúng** tilt của mọi node trong cây con đó (mỗi node đúng một lần).

Quy nạp:
- `node is None`: trả 0, không node nào ✔.
- Giả sử đúng cho hai con: `L`, `R` là tổng đúng của hai cây con, và tilt mọi node trong hai cây con đã được cộng. Ta cộng thêm `|L − R|` = tilt của chính `node` (đúng định nghĩa), và trả `node.val + L + R` = tổng cây con gốc `node` ✔.

Áp dụng cho root ⇒ `total` = tổng tilt của mọi node.

## 10. So sánh và độ phức tạp

Gọi `n` = số node, `h` = chiều cao.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Baseline gọi `subtree_sum` lại | `O(n²)` xấu nhất, `O(n log n)` nếu cân bằng | `O(h)` | dịch thẳng định nghĩa |
| 2. DFS post-order một lần | `O(n)` | `O(h)` call stack | **khuyên dùng** |
| 2b. Trả về cặp (không `nonlocal`) | `O(n)` | `O(h)` | sạch, dễ test |
| 3. Post-order lặp bằng stack | `O(n)` | `O(n)` (dict + stack) | không lo RecursionError |

`h = O(log n)` với cây cân bằng; `O(n)` với cây lệch.

## 11. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

In [16]:
tests = [
    ([1, 2, 3], 1),
    ([4, 2, 9, 3, 5, null, 7], 15),
    ([21, 7, 14, 1, 1, 2, 2, 3, 3], 9),
    ([], 0),
    ([5], 0),
    ([1, 2], 2),
    ([1, null, 2], 2),
    ([-1, -2, 3], 5),             # giá trị âm: |(-2) - 3| = 5
    ([0, 0, 0, 0, 0], 0),
]
methods = [find_tilt_naive, find_tilt_dfs, find_tilt_pair, find_tilt_iterative]
for values, expected in tests:
    t = build_tree(values)
    for f in methods:
        assert f(t) == expected, (f.__name__, values, f(t))
    print(f"{str(values):<32} -> {expected}")
print("Test mẫu PASS!")

[1, 2, 3]                        -> 1
[4, 2, 9, 3, 5, None, 7]         -> 15
[21, 7, 14, 1, 1, 2, 2, 3, 3]    -> 9
[]                               -> 0
[5]                              -> 0
[1, 2]                           -> 2
[1, None, 2]                     -> 2
[-1, -2, 3]                      -> 5
[0, 0, 0, 0, 0]                  -> 0
Test mẫu PASS!


In [17]:
import random

def random_tree(n: int) -> Optional[TreeNode]:
    if n == 0:
        return None
    nodes = [TreeNode(random.randint(-1000, 1000))]
    for _ in range(1, n):
        while True:
            p = random.choice(nodes)
            side = random.choice(("left", "right"))
            if getattr(p, side) is None:
                child = TreeNode(random.randint(-1000, 1000))
                setattr(p, side, child)
                nodes.append(child)
                break
    return nodes[0]

random.seed(563)
for _ in range(300):
    t = random_tree(random.randint(0, 40))
    answers = {f(t) for f in methods}
    assert len(answers) == 1, answers
print("300 cây ngẫu nhiên: 4 cách cho cùng kết quả.")

300 cây ngẫu nhiên: 4 cách cho cùng kết quả.


`getattr(p, "left")` đọc thuộc tính theo tên chuỗi, `setattr(p, "left", child)` gán thuộc tính theo tên — tiện để chọn trái/phải ngẫu nhiên.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ dùng giá trị con trực tiếp thay vì tổng cả cây con

In [18]:
def wrong_children_only(root):
    if root is None:
        return 0
    lv = root.left.val if root.left else 0
    rv = root.right.val if root.right else 0
    return abs(lv - rv) + wrong_children_only(root.left) + wrong_children_only(root.right)

print("sai:", wrong_children_only(ex2), "| đúng:", find_tilt_dfs(ex2))

sai: 16 | đúng: 15


Vì sao sai: tại root 4, code sai tính `|2 − 9| = 7` thay vì `|10 − 16| = 6`. Đề yêu cầu tổng **mọi** node trong cây con.

### Lỗi 2: Hàm đệ quy trả về tilt thay vì tổng

Nếu `dfs` trả về tilt (hoặc `total`), cha sẽ nhận sai `L`, `R`. Hàm phải trả **tổng cây con**; tilt chỉ được cộng vào biến ngoài.

In [19]:
def wrong_return_tilt(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return 0
        L, R = dfs(node.left), dfs(node.right)
        tilt = abs(L - R)
        total += tilt
        return tilt                 # SAI: phải trả node.val + L + R
    dfs(root)
    return total

print("sai:", wrong_return_tilt(ex2), "| đúng:", find_tilt_dfs(ex2))

sai: 0 | đúng: 15


### Lỗi 3: Quên cộng `node.val` vào tổng trả về

`return L + R` bỏ mất giá trị của chính node → tổng cây con sai từ tầng thứ hai trở lên.

### Lỗi 4: Quên `abs`

Hiệu có thể âm → tổng sai (thậm chí âm).

In [20]:
def wrong_no_abs(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return 0
        L, R = dfs(node.left), dfs(node.right)
        total += L - R               # SAI: thiếu abs
        return node.val + L + R
    dfs(root)
    return total

print("sai:", wrong_no_abs(ex2), "| đúng:", find_tilt_dfs(ex2))

sai: -15 | đúng: 15


### Lỗi 5: Dùng `self.total` mà không reset

Nếu lưu `self.total` làm thuộc tính và LeetCode gọi `findTilt` nhiều lần trên cùng object, kết quả bị cộng dồn. Luôn khởi tạo lại ở đầu hàm chính (hoặc dùng biến cục bộ + `nonlocal` như trên).

## 13. Mẹo nhận dạng pattern và biến thể

### Mẹo
- Đại lượng của node phụ thuộc vào **tổng/chiều cao/... của cây con** → **post-order**: hàm trả về thông tin cây con cho cha, đáp án gom vào biến ngoài.
- Baseline gọi lại hàm con cho mỗi node → dấu hiệu gộp thành một lần duyệt.
- Nhãn "DP on Trees" của LeetCode: mỗi node tính từ kết quả của con — chính là post-order.

### Biến thể
1. **LeetCode 543 – Diameter of Binary Tree**: trả về chiều cao, gom `L + R`.
2. **LeetCode 110 – Balanced Binary Tree**: trả về chiều cao, kiểm tra `|L − R| ≤ 1`.
3. **LeetCode 508 – Most Frequent Subtree Sum**: trả về tổng, đếm tần suất bằng dict.
4. **LeetCode 1120 – Maximum Average Subtree**: trả về `(tổng, số node)`.
5. **LeetCode 2265 – Count Nodes Equal to Average of Subtree**.

In [21]:
# Biến thể 3 demo: tổng cây con xuất hiện nhiều nhất
from collections import Counter

def most_frequent_subtree_sum(root):
    freq = Counter()
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        freq[s] += 1
        return s
    dfs(root)
    best = max(freq.values())
    return sorted(s for s, c in freq.items() if c == best)

print(most_frequent_subtree_sum(build_tree([5, 2, -3])))   # [-3, 2, 4]
print(most_frequent_subtree_sum(build_tree([5, 2, -5])))   # [2]

[-3, 2, 4]
[2]


`Counter` là dict đặc biệt: truy cập key chưa có trả về 0, nên `freq[s] += 1` không cần kiểm tra trước.

## 14. Bài tập tự luyện

### Bài 1 (dễ)
Viết `node_tilt(node)` tính độ nghiêng của **một** node (dùng `subtree_sum`).

In [22]:
def node_tilt(node) -> int:
    # TODO
    pass

# assert node_tilt(ex2) == 6
# assert node_tilt(ex2.right) == 7

### Bài 2 (dễ)
Viết `all_subtree_sums(root)` trả về list tổng cây con của mọi node theo thứ tự **post-order**.

In [23]:
def all_subtree_sums(root) -> list:
    # TODO
    pass

# assert all_subtree_sums(ex2) == [3, 5, 10, 7, 16, 30]

### Bài 3 (vừa)
Viết `max_tilt_node(root)` trả về **giá trị** của node có tilt lớn nhất (nếu bằng nhau, lấy node hoàn thành trước theo post-order).

In [24]:
def max_tilt_node(root):
    # TODO
    pass

# assert max_tilt_node(ex2) == 9
# assert max_tilt_node(ex3) == 7

### Bài 4 (vừa)
Viết `tilt_tree(root)`: trả về **cây mới** cùng hình dạng, mỗi node mang giá trị là tilt của node tương ứng.

In [25]:
def tilt_tree(root):
    # TODO
    pass

# Gợi ý kiểm tra: dùng hàm to_level_list ở phần đáp án
# to_level_list(tilt_tree(ex2)) == [6, 2, 7, 0, 0, None, 0]

### Bài 5 (khó hơn)
Viết `is_balanced_by_sum(root, k)`: trả `True` nếu **mọi** node có tilt ≤ `k`. Dừng sớm khi phát hiện vi phạm (gợi ý: trả về `None` để báo vi phạm lên cha).

In [26]:
def is_balanced_by_sum(root, k: int) -> bool:
    # TODO
    pass

# assert is_balanced_by_sum(ex2, 7) == True
# assert is_balanced_by_sum(ex2, 6) == False
# assert is_balanced_by_sum(None, 0) == True

## 15. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Cần tổng trái, tổng phải | `subtree_sum(node.left)` | `abs(...)` |
| 2 | Post-order: con trước | Hàm trả tổng, `append` sau khi tính | List bên ngoài + `append` |
| 3 | Giống DFS chính | Theo dõi `(best_tilt, best_val)` | So sánh `>` (không `>=`) để giữ node trước |
| 4 | Đệ quy trả về `(tổng, node_mới)` | Dựng node con trước | `TreeNode(abs(L-R), new_left, new_right)` |
| 5 | Trả `None` khi vi phạm | Con trả `None` → cha trả `None` ngay | Kết quả cuối: `solve(root) is not None` |

## 16. Đáp án tham khảo (tự làm trước rồi mới xem)

In [27]:
def to_level_list(root) -> list:
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def answer_node_tilt(node) -> int:
    return abs(subtree_sum(node.left) - subtree_sum(node.right))

def answer_all_subtree_sums(root) -> list:
    res = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        res.append(s)
        return s
    dfs(root)
    return res

def answer_max_tilt_node(root):
    best = [-1, None]
    def dfs(node):
        if node is None:
            return 0
        L, R = dfs(node.left), dfs(node.right)
        if abs(L - R) > best[0]:
            best[0], best[1] = abs(L - R), node.val
        return node.val + L + R
    dfs(root)
    return best[1]

def answer_tilt_tree(root):
    def dfs(node):
        if node is None:
            return 0, None
        L, new_left = dfs(node.left)
        R, new_right = dfs(node.right)
        return node.val + L + R, TreeNode(abs(L - R), new_left, new_right)
    return dfs(root)[1]

def answer_is_balanced_by_sum(root, k: int) -> bool:
    def solve(node):
        if node is None:
            return 0
        L = solve(node.left)
        if L is None:
            return None
        R = solve(node.right)
        if R is None:
            return None
        if abs(L - R) > k:
            return None
        return node.val + L + R
    return solve(root) is not None

assert answer_node_tilt(ex2) == 6
assert answer_node_tilt(ex2.right) == 7
assert answer_all_subtree_sums(ex2) == [3, 5, 10, 7, 16, 30]
assert answer_max_tilt_node(ex2) == 9
assert answer_max_tilt_node(ex3) == 7
assert to_level_list(answer_tilt_tree(ex2)) == [6, 2, 7, 0, 0, None, 0]
assert answer_is_balanced_by_sum(ex2, 7) == True
assert answer_is_balanced_by_sum(ex2, 6) == False
assert answer_is_balanced_by_sum(None, 0) == True
for _ in range(100):
    t = random_tree(random.randint(0, 30))
    tt = answer_tilt_tree(t)
    assert subtree_sum(tt) == find_tilt_dfs(t)     # tổng cây tilt = đáp án bài gốc
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


## 17. Template nhớ nhanh

```python
class Solution:
    def findTilt(self, root: Optional[TreeNode]) -> int:
        total = 0
        def dfs(node):
            nonlocal total
            if not node:
                return 0
            L, R = dfs(node.left), dfs(node.right)
            total += abs(L - R)          # gom đáp án
            return node.val + L + R      # trả tổng cây con cho cha
        dfs(root)
        return total
```

Khuôn chung "post-order trả một thứ, gom một thứ khác" dùng lại cho 543, 110, 508, 124...

## 18. Checklist trước khi submit

- [ ] "Tổng cây con" là tổng **mọi** node bên dưới, không chỉ con trực tiếp.
- [ ] Hàm đệ quy trả về **tổng** (`node.val + L + R`), không trả tilt.
- [ ] Có `abs(L - R)`.
- [ ] Cây rỗng trả 0.
- [ ] Có `nonlocal total` (hoặc trả về tuple).
- [ ] Nghĩ tới cây lệch `10^4` node → bản lặp hoặc tăng giới hạn đệ quy.